# Hugging Face Veri Seti Kalite Analizi

## tl;dr

- Dokuz veri setinde toplam **3.119 satır** incelendi; 3.016 sohbet kaydı ve 103 katalog kaydı eksiksiz profillendi.
- Yapısal bütünlük güçlü: sohbet setlerinde boş içerik, geçersiz rol veya birebir satır tekrarı bulunmadı. Buna rağmen Marvel ve felsefe istemlerinde, biyoloji ve Şahin kimlik cevaplarında yüksek normalleştirilmiş tekrar yoğunluğu var.
- Üç veri setindeki **1.136** asistan mesajında ayrı `thinking` alanı bulunuyor. Şahin kimlik setinde **462** alan gerçek `null` yerine metin olarak `"null"` içeriyor.
- MEB verisi zaman hassasiyeti nedeniyle güncel kaynak katmanı gerektiriyor. İthaki kataloğu anahtar ve fiyat kontrollerini geçiyor; ancak bazı katalog alanları seyrek. Genel bilgi ve e-ticaret setleri düzenli olsa da sentetik doğruluk ve politika iddiaları doğrulanmalı.

## Bağlam ve Yöntem

Dokuz herkese açık Hugging Face veri setinin sabit sürüm kopyaları 20 Temmuz 2026 tarihinde indirildi. Profil hesapları 21 Temmuz 2026 tarihinde yerel ham satırlar üzerinden yeniden çalıştırıldı ve önceki JSON çıktılarıyla birebir aynı sonuç verdi. Şema, tamamlılık, rol sırası, birebir ve normalleştirilmiş tekrar, metin uzunluğu, `thinking` alanı, kişisel veri biçimleri, zaman hassasiyeti, katalog anahtarları ve fiyat tutarlılığı kontrolleri uygulandı.

### Temel Varsayımlar

- Düzenli ifade (regex) tabanlı kişisel veri taraması, isim ve bağlamsal kimlik gibi yapılandırılmamış kişisel veriyi bütünüyle yakalayamaz.
- Normalleştirilmiş tekrar oranında her tekrar ailesinin ilk örneği korunur; sonraki ek kopyalar toplam satır sayısına bölünür. Bu ölçüm anlamsal benzerliğin tamamını kapsamaz.
- Zaman hassasiyeti sayıları, düzenli ifadeyle bulunan eşleşme adetleridir; benzersiz satır sayısı olarak yorumlanmamalıdır.
- Olgusal doğruluk için tam uzman incelemesi yapılmadı; riskler örnek içerik, kaynak yöntemi ve mevcut kaynak izleme alanlarına göre raporlandı.

## Veri

In [1]:
import json
from pathlib import Path
import pandas as pd

WORKING_DIR = Path.cwd()
PROJECT_DIR = next(
    (
        candidate
        for candidate in (WORKING_DIR, WORKING_DIR.parent)
        if (candidate / "outputs" / "data_quality_profiles.json").exists()
    ),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Project outputs were not found in the working directory or its parent."
    )
profiles = json.loads((PROJECT_DIR / "outputs" / "data_quality_profiles.json").read_text(encoding="utf-8"))
manual = json.loads((PROJECT_DIR / "outputs" / "manual_findings.json").read_text(encoding="utf-8"))
overlaps = json.loads((PROJECT_DIR / "outputs" / "cross_dataset_overlap.json").read_text(encoding="utf-8"))

len(profiles), sum(item["profile"]["row_count"] for item in profiles)

(9, 3119)

In [2]:
summary_rows = []
for item in profiles:
    profile = item["profile"]
    scan = profile.get("text_scan", {})
    summary_rows.append({
        "dataset": item["dataset_id"],
        "satır": profile["row_count"],
        "biçim": profile["data_shape"],
        "birebir_tekrar": profile["exact_duplicate_rows"],
        "istem_tekrar_%": round(100 * profile.get("user_prompt_duplicates", {}).get("duplicate_rate", 0), 1),
        "cevap_tekrar_%": round(100 * profile.get("assistant_answer_duplicates", {}).get("duplicate_rate", 0), 1),
        "thinking": profile.get("nonempty_thinking_messages", 0),
        "metin_null": profile.get("string_encoded_null_fields", 0),
        "zaman_eşleşmesi": scan.get("time_sensitive_matches", 0),
        "asistan_kelime_medyan": profile.get("assistant_word_length", {}).get("median"),
        "asistan_kelime_p95": profile.get("assistant_word_length", {}).get("p95"),
    })

summary = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
summary

,dataset,satır,biçim,birebir_tekrar,istem_tekrar_%,cevap_tekrar_%,thinking,metin_null,zaman_eşleşmesi,asistan_kelime_medyan,asistan_kelime_p95
0,Aysenur44/namaz-vakti-identity-tr,4,conversation,0,0.0,0.0,0,0,0,6.0,10.70
1,Egertekin/marvel-domain-dataset,177,conversation,0,75.7,13.6,0,0,1,56.0,153.40
2,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,186,conversation,0,0.0,0.0,186,0,31,87.0,119.00
3,aliFurkan123/cultural-questions-dataset,500,conversation,0,0.0,0.0,500,0,4,20.0,24.00
4,gururaser/ithaki-bilimkurgu-klasikleri,103,catalog,0,0.0,0.0,0,0,0,NaN,NaN
5,namruni/meb-ogretmen-soru-cevap,450,conversation,0,0.0,0.0,450,0,297,81.0,114.55
6,nyzmemre/biyoloji-terimleri-turkce-sa,1093,conversation,0,0.2,21.6,0,0,0,15.0,41.00
7,sk75/sahin_identity,77,conversation,0,0.0,64.9,0,462,0,11.0,26.00
8,yoitsmeyusuf/felsefe_finetune,529,conversation,0,76.6,0.0,0,0,21,35.0,353.60


## Sonuçlar

### Portföy toplamları ve yapısal kontroller doğrulandı

Aşağıdaki hücre, rapordaki temel toplamları doğrudan profillerden yeniden hesaplar ve beklenen değerlerle doğrular. Ham büyüklük tek başına kalite göstergesi değildir.

In [3]:
portfolio = pd.Series({
    "veri_seti": len(summary),
    "toplam_satır": int(summary["satır"].sum()),
    "sohbet_satırı": int(summary.loc[summary["biçim"] == "conversation", "satır"].sum()),
    "katalog_satırı": int(summary.loc[summary["biçim"] == "catalog", "satır"].sum()),
    "thinking_içeren_mesaj": int(summary["thinking"].sum()),
    "metin_null_hatası": int(summary["metin_null"].sum()),
    "zaman_hassasiyeti_eşleşmesi": int(summary["zaman_eşleşmesi"].sum()),
}, name="değer")
assert portfolio["veri_seti"] == 9
assert portfolio["toplam_satır"] == 3119
assert portfolio["sohbet_satırı"] == 3016
assert portfolio["katalog_satırı"] == 103
assert portfolio["thinking_içeren_mesaj"] == 1136
assert portfolio["metin_null_hatası"] == 462
portfolio.to_frame()

,değer
veri_seti,9
toplam_satır,3119
sohbet_satırı,3016
katalog_satırı,103
thinking_içeren_mesaj,1136
metin_null_hatası,462
zaman_hassasiyeti_eşleşmesi,354


### Tekrar ve şema riskleri veri setine göre yoğunlaşıyor

Marvel ve felsefe setlerinde aynı kullanıcı sorusu çok sayıda farklı cevapla eşleşiyor. Biyoloji veri setinde aynı tanım farklı istemlerle yineleniyor. Şahin kimlik setinde ise hem cevap tekrarı hem de metin olarak saklanan `null` değerleri öne çıkıyor.

In [4]:
summary[["dataset", "satır", "istem_tekrar_%", "cevap_tekrar_%", "thinking", "metin_null"]]

,dataset,satır,istem_tekrar_%,cevap_tekrar_%,thinking,metin_null
0,Aysenur44/namaz-vakti-identity-tr,4,0.0,0.0,0,0
1,Egertekin/marvel-domain-dataset,177,75.7,13.6,0,0
2,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,186,0.0,0.0,186,0
3,aliFurkan123/cultural-questions-dataset,500,0.0,0.0,500,0
4,gururaser/ithaki-bilimkurgu-klasikleri,103,0.0,0.0,0,0
5,namruni/meb-ogretmen-soru-cevap,450,0.0,0.0,450,0
6,nyzmemre/biyoloji-terimleri-turkce-sa,1093,0.2,21.6,0,0
7,sk75/sahin_identity,77,0.0,64.9,0,462
8,yoitsmeyusuf/felsefe_finetune,529,76.6,0.0,0,0


### Yanıt uzunluğu veri setleri arasında belirgin biçimde değişiyor

Medyan ve yüzde 95 (p95) asistan yanıt uzunlukları, kısa sözlük açıklamalarından uzun görüş ve mevzuat yanıtlarına kadar farklı kullanım biçimlerini gösteriyor. Bu değerler bağlam penceresi, örnek ağırlığı ve hedef görev tasarımı için ölçüdür.

In [5]:
summary.loc[
    summary["biçim"] == "conversation",
    ["dataset", "asistan_kelime_medyan", "asistan_kelime_p95"],
].sort_values("asistan_kelime_medyan", ascending=False)

,dataset,asistan_kelime_medyan,asistan_kelime_p95
2,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,87.0,119.00
5,namruni/meb-ogretmen-soru-cevap,81.0,114.55
1,Egertekin/marvel-domain-dataset,56.0,153.40
8,yoitsmeyusuf/felsefe_finetune,35.0,353.60
3,aliFurkan123/cultural-questions-dataset,20.0,24.00
6,nyzmemre/biyoloji-terimleri-turkce-sa,15.0,41.00
7,sk75/sahin_identity,11.0,26.00
0,Aysenur44/namaz-vakti-identity-tr,6.0,10.70


### Katalog veri seti anahtar ve fiyat kontrollerini geçiyor

İthaki kataloğunda ISBN, kitap URL'si ve kitap–yazar anahtarı benzersiz; ISBN-13 ve fiyat/indirim hesapları tutarlı. En belirgin veri kalitesi konusu bazı alanların yüksek boşluk oranı ile fiyat ve özetlerin zaman/kaynak meta verisi eksikliğidir.

In [6]:
catalog = next(item["profile"] for item in profiles if item["profile"]["data_shape"] == "catalog")
pd.DataFrame({
    "kontrol": [
        "satır", "birebir tekrar", "ISBN tekrarı", "geçersiz ISBN-13",
        "URL tekrarı", "geçersiz URL", "indirim tutarsızlığı",
        "eksik çevirmen", "eksik kapak tipi", "eksik orijinal ad", "eksik yayın tarihi",
    ],
    "değer": [
        catalog["row_count"], catalog["exact_duplicate_rows"], catalog["duplicate_isbn_rows"],
        catalog["invalid_isbn13_rows"], catalog["duplicate_book_url_rows"], catalog["invalid_url_rows"],
        catalog["discount_inconsistency_rows"], catalog["null_counts"]["cevirmen"],
        catalog["null_counts"]["kapak_tipi"], catalog["null_counts"]["orijinal_adi"],
        catalog["null_counts"]["yayin_tarihi"],
    ],
})

,kontrol,değer
0,satır,103
1,birebir tekrar,0
2,ISBN tekrarı,0
3,geçersiz ISBN-13,0
4,URL tekrarı,0
5,geçersiz URL,0
6,indirim tutarsızlığı,0
7,eksik çevirmen,62
8,eksik kapak tipi,96
9,eksik orijinal ad,65


### Veri setleri arası ortak şablon var

NamazAsistan ve Şahin kimlik veri setleri dört kullanıcı istemini paylaşıyor. Bu ortaklık, kimlik tohumları birlikte kullanılacaksa kaynak şablonun belgelenmesini ve ortak istemlerin tek bir şablon ailesi olarak etiketlenmesini gerektirir.

In [7]:
pd.DataFrame(overlaps)

,left,right,shared_user_prompts,shared_assistant_answers,prompt_examples,answer_examples
0,Aysenur44/namaz-vakti-identity-tr,sk75/sahin_identity,4,0,"[adın ne, ne işe yararsın, sen kimsin]",[]


### Bulguların önem düzeyi ve önerilen düzeltmeler

Aşağıdaki tablo, her veri setinin en önemli veri kalite risklerini ve en küçük etkili düzeltmeyi özetler. Değerlendirme, kullanım amacına göre engelleri görünür kılar.

In [8]:
finding_rows = []
for dataset_id, detail in manual["datasets"].items():
    for finding in detail["findings"]:
        finding_rows.append({
            "dataset": dataset_id,
            "önem": finding["severity"],
            "kanıt": finding["evidence"],
            "risk": finding["risk"],
            "düzeltme": finding["remediation"],
        })
findings = pd.DataFrame(finding_rows)
findings

,dataset,önem,kanıt,risk,düzeltme
0,aliFurkan123/cultural-questions-dataset,high,500 asistan mesajının tamamında thinking alanı...,Sentetik olgusal hatalar ve gerekçe hataları b...,"Bağımsız olgu kontrolü, kaynak/citation alanı ..."
1,aliFurkan123/cultural-questions-dataset,medium,Depo adı kültürel soruları ima ederken veri ka...,Kapsam yanlış anlaşılabilir ve kategori denges...,"Depo adını/kartını eşleştir; konu, zorluk ve k..."
2,Aysenur44/namaz-vakti-identity-tr,critical,Yalnız 4 satır var ve bunlar kimlik/yetenek so...,Alan uzmanlığı kazandırmaz; model yalnız isim ...,"Doğrulanmış ibadet içeriği, konum-zaman bağlam..."
3,Aysenur44/namaz-vakti-identity-tr,high,"Kaynak, amaç, sınırlama ve örnek kullanım açık...",Hedeflenen davranış ve veri kökeni belirsiz.,"Kaynak, amaç, sınırlama ve örnek kullanım içer..."
4,Aysenur44/namaz-vakti-identity-tr,medium,Dört kullanıcı isteminin tamamı sk75/sahin_ide...,Şablon tekrarına ve kimlik davranışının aşırı ...,Kaynak şablonu belirt; ortak istemleri tek bir...
5,Egertekin/marvel-domain-dataset,high,177 kullanıcı sorusunun 134'ü normalleştirilmi...,Spider-Man ve birkaç genel soru eğitim sinyali...,Her paragraf için özgül soru üret; karakter/ko...
6,Egertekin/marvel-domain-dataset,high,Kaynağın Wikipedia olduğu yazıyor fakat sayfa ...,Yanıtların hangi kaynak sürümünden türetildiği...,Kaynak URL'sini ve revizyon kimliğini satır dü...
7,Egertekin/marvel-domain-dataset,medium,Veri kartı instruction/input/output sütunların...,Kullanıcılar yanlış yükleme kodu veya şema var...,Kartı gerçek şemayla eşleştir ve yükleme örneğ...
8,gururaser/ithaki-bilimkurgu-klasikleri,high,"Veriler yayıncı sitesinden, kapak görselleri i...",Kaynak değiştiğinde metin ve görsel kayıtların...,"Toplama zamanı, kaynak türü ve görsel kaynağı ..."
9,gururaser/ithaki-bilimkurgu-klasikleri,medium,"Çevirmen 62, kapak tipi 96, orijinal ad 65 ve ...",Alan bazlı analiz ve filtreleme kapsaması deği...,Eksik nedenini belirten durum alanı ekle ve al...


## Çıkarımlar

- **Doğrudan kullanılabilirlik yerine kullanım koşulu belirlemek gerekiyor.** Her veri seti farklı bir göreve yarıyor; kaynak izlenebilirliği, zaman hassasiyeti ve tekrar yoğunluğu görev bazında ele alınmalı.
- **Yapısal temizlik güçlü fakat normalleştirilmiş tekrar kritik.** Birebir tekrarların sıfır olması, aynı sorunun veya cevabın onlarca kez yinelenmesini engellemiyor.
- **Yüksek riskli alanlar kaynak bağlı olmalı.** MEB ve e-ticaret içerikleri, güncel politika veya mevzuat getiren retrieval (kaynak getirme) katmanı ve açık güven sınırları olmadan kesin yanıt üretiminde kullanılmamalı.
- **Katalog verisi diğerlerinden farklı ele alınmalı.** İthaki seti sohbet değil yapılandırılmış ürün kataloğu; toplama zamanı ve kaynak meta verisi tanımlanırsa analiz ve kontrollü üretim için değerlidir.
- **Kimlik setleri ana eğitim verisi değildir.** Dört ve 77 satırlık kimlik örnekleri düşük ağırlıklı davranış tohumu olarak tutulmalı; alan yetkinliği veya genel kalite kanıtı sayılmamalı.